# Week 6-3 — Baseline vs Agentic 답변 분석

두 시스템의 답변 차이를 발생 원인별로 나눠 추출하고, 정성 판정용 파일을 생성한다.
판정은 생성된 마크다운 파일에서 직접 수행한다.

| 구분 | 문항 수 | 성격 |
|---|---|---|
| 검색·답변 모두 동일 | 15 | 비교 불필요 |
| 검토 1 — 검색 동일, 답변 다름 | 17 | 구조 무관. API 비결정성 |
| 검토 2 — 거절 문구 차이 | 4 | `refuse` 노드 개입 |
| 검토 3 — 재검색 | 5 | Agentic 구조가 실제 개입 |
| 검토 4 — 예외 | 1 | 검색 결과 자체가 미세하게 다름 |

**판정 기준**

- 층 A(검토 2·3, 9문항) — Agentic 구조의 성능 판단 대상. 개선 / 동등 / 악화
- 층 B(검토 1, 17문항) — 성능 비교 대상이 아님. API 변동 폭 측정용

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.rag import config

P = config.DATA_DIR / "processed"
a = pd.read_csv(P / "week6_agentic_v2_result.csv")
b = pd.read_csv(P / "week6_baseline_v2_result.csv")

m = a.merge(
    b[["qid", "answer", "max_score", "latency", "contexts", "citations"]],
    on="qid", suffixes=("_a", "_b")
)
print("병합 완료:", len(m), "문항")

병합 완료: 42 문항


## 검토 1 — 검색은 같은데 답변이 다른 문항 (17개)

재검색을 하지 않았고 `contexts`가 완전히 동일한데 답변 문장이 달라진 문항.
두 시스템이 같은 입력으로 같은 함수(`generate`)를 호출했으므로 **구조 차이로는 설명되지 않는다.**
거절 문구 차이 4건(32·33·37·38)은 `refuse` 노드의 고정 문구 때문이므로 제외했다.

**검토 관점** — 표현만 다른가, 전달하는 정보 자체가 다른가

→ `data/processed/week6_answer_diff_review.md`

In [ ]:
TARGETS = [2, 3, 6, 7, 8, 9, 11, 13, 14, 17, 18, 19, 22, 25, 39, 41, 42]

lines = []
for q in TARGETS:
    r = m[m.qid == q].iloc[0]
    lines.append(f"## qid {q} — {r.q_type} / {r.lang} / score {r.max_score_a:.3f}\n")
    lines.append(f"**Q.** {r.question}\n")
    lines.append(f"### Baseline\n{r.answer_b}\n")
    lines.append(f"### Agentic\n{r.answer_a}\n")
    lines.append("**분류**: [ ] 표현만 다름  [ ] 내용이 다름  [ ] 한쪽이 나음\n")
    lines.append("---\n")

out = P / "week6_answer_diff_review.md"
out.write_text("\n".join(lines), encoding="utf-8")
print("저장:", out)

## 검토 2 — 거절 문구 차이 (4개)

In [3]:
TARGETS = [32, 33, 37, 38]

lines = ["# Week 6 검토 2 — 거절 문구 차이\n",
         "Agentic은 `refuse` 노드의 고정 문구, Baseline은 LLM 생성 문구.\n",
         "**검토 관점**: 거절이 정당했는가 / 고정 문구가 LLM 문구보다 나은가 / 언어가 질문과 일치하는가\n",
         "---\n"]

for q in TARGETS:
    r = m[m.qid == q].iloc[0]
    lines += [
        f"## qid {q} — {r.q_type} / {r.lang}",
        f"- 기대 행동: `{r.expected_behavior}` · Agentic decision: `{r.decision}`",
        f"- max_score: {r.max_score_b:.3f} (baseline) / {r.max_score_a:.3f} (agentic)",
        f"- latency: {r.latency_b:.2f}s → {r.latency_a:.2f}s\n",
        f"**Q.** {r.question}\n",
        f"### Baseline\n{r.answer_b}\n",
        f"### Agentic\n{r.answer_a}\n",
        "**판정**: [ ] 개선  [ ] 동등  [ ] 악화",
        "**거절 정당성**: [ ] 정당  [ ] 부당",
        "**메모**: \n",
        "---\n",
    ]

out = P / "week6_review_2_refusal.md"
out.write_text("\n".join(lines), encoding="utf-8")
print("저장:", out)

저장: /Users/jian/Documents/rag-agent-portfolio/data/processed/week6_review_2_refusal.md


## 검토 3 — 재검색을 탄 문항 (5개)

In [4]:
TARGETS = [4, 10, 31, 34, 36]

lines = ["# Week 6 검토 3 — 재검색(rewrite)을 탄 문항\n",
         "Agentic 구조가 실제로 개입한 문항. Agentic의 존재 이유를 판단하는 대상.\n",
         "**검토 관점**: 재작성 질문이 원 의도를 유지했는가 / 점수 상승이 답변 개선으로 이어졌는가 / 재검색할 가치가 있었는가\n",
         "---\n"]

for q in TARGETS:
    r = m[m.qid == q].iloc[0]
    delta = r.max_score_a - r.max_score_b
    lines += [
        f"## qid {q} — {r.q_type} / {r.lang}",
        f"- 기대 행동: `{r.expected_behavior}` · Agentic decision: `{r.decision}` · 재시도 {r.retry_count}회",
        f"- max_score: {r.max_score_b:.3f} → {r.max_score_a:.3f} ({delta:+.3f})",
        f"- latency: {r.latency_b:.2f}s → {r.latency_a:.2f}s\n",
        f"**Q.** {r.question}\n",
        "### 실행 경로",
        "```",
        str(r.route).replace(" → ", "\n→ "),
        "```\n",
        f"**정답 요지**\n{r.ground_truth}\n",
        f"### Baseline\n{r.answer_b}\n",
        f"### Agentic\n{r.answer_a}\n",
        "**판정**: [ ] 개선  [ ] 동등  [ ] 악화",
        "**재작성 질문이 의도 유지**: [ ] 예  [ ] 아니오",
        "**재검색할 가치**: [ ] 있었음  [ ] 없었음",
        "**메모**: \n",
        "---\n",
    ]

out = P / "week6_review_3_rewrite.md"
out.write_text("\n".join(lines), encoding="utf-8")
print("저장:", out)

저장: /Users/jian/Documents/rag-agent-portfolio/data/processed/week6_review_3_rewrite.md


## 검토 4 — 예외 문항 (1개)

In [5]:
TARGETS = [23]

lines = ["# Week 6 검토 4 — 예외 문항\n",
         "재검색을 하지 않았고 max_score도 동일한데 `contexts`가 다른 문항.\n",
         "1등 조각은 같으나 2~5위 구성 또는 순서가 달랐다는 뜻이며, 검색 단계의 비결정성 가능성을 시사한다.\n",
         "---\n"]

for q in TARGETS:
    r = m[m.qid == q].iloc[0]
    cb = str(r.contexts_b).split("\n---\n")
    ca = str(r.contexts_a).split("\n---\n")
    lines += [
        f"## qid {q} — {r.q_type} / {r.lang}",
        f"- max_score: {r.max_score_b:.3f} / {r.max_score_a:.3f} (동일)",
        f"- 검색 조각 수: baseline {len(cb)}개 / agentic {len(ca)}개\n",
        f"**Q.** {r.question}\n",
        "### 출처 비교",
        f"- Baseline: `{r.citations_b}`",
        f"- Agentic : `{r.citations_a}`\n",
        "### 검색 조각 대조 (앞 200자)\n",
    ]
    for i in range(max(len(cb), len(ca))):
        tb = cb[i][:200].replace("\n", " ") if i < len(cb) else "(없음)"
        ta = ca[i][:200].replace("\n", " ") if i < len(ca) else "(없음)"
        lines += [
            f"**[{i+1}] {'동일' if tb == ta else '다름'}**",
            f"- B: {tb}",
            f"- A: {ta}\n",
        ]
    lines += [
        f"### Baseline 답변\n{r.answer_b}\n",
        f"### Agentic 답변\n{r.answer_a}\n",
        "**판정**: [ ] 개선  [ ] 동등  [ ] 악화",
        "**검색 차이 원인 추정**: \n",
        "**메모**: \n",
        "---\n",
    ]

out = P / "week6_review_4_other.md"
out.write_text("\n".join(lines), encoding="utf-8")
print("저장:", out)

저장: /Users/jian/Documents/rag-agent-portfolio/data/processed/week6_review_4_other.md
